<img src="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers-logo.png" width="500" align="center">

# Transfer Learning для NLP: BERT - как одна модель изменила всё

**Роль:** Преподаватель по ML
**Уровень:** средний (основы PyTorch + обработка текста)
**Время прохождения:** ~120-150 минут

---

### Чему вы научитесь

После прохождения этого ноутбука вы:
- поймёте, **зачем** нужен transfer learning для NLP;
- разберёте архитектуру **BERT** - от токенизации до выхода;
- увидите, как работает **WordPiece** токенизация;
- загрузите предобученную модель **HuggingFace** и изучите её;
- визуализируете **вложения (embeddings)** BERT через t-SNE;
- поэкспериментируете с **Masked Language Model** интерактивно;
- **дообучите** BERT для классификации текста;
- оцените модель и визуализируете **confusion matrix**;
- заглянете внутрь **attention heads** и поймёте, что учит BERT;
- получите практические **советы** по тонкой настройке BERT.

### Принцип этого блокнота

> Вы - автор, не пользователь. Каждая строка видна. Каждое действие можно сломать и починить. Никаких "чёрных ящиков".

---

## План урока

| # | Шаг | Что делаем |
|---|-----|------------|
| 1 | Подготовка окружения | Установка и импорт библиотек |
| 2 | Что такое BERT? | Двунаправленный энкодер, предобучение (MLM + NSP) |
| 3 | Токенизатор BERT: WordPiece | Как BERT разбивает текст на токены |
| 4 | Архитектура BERT: глубокий разбор | Embedding, Encoder слои, выход |
| 5 | Загрузка предобученного BERT | HuggingFace step-by-step |
| 6 | Вложения BERT: исследование | Визуализация токенных embeddings |
| 7 | Masked Language Model: предсказание [MASK] | Интерактивный демо |
| 8 | Fine-tuning BERT для классификации | Датасет, цикл обучения |
| 9 | Оценка дообученного BERT | Метрики, confusion matrix |
| 10 | BERT vs предыдущие подходы | Сравнительная таблица |
| 11 | Что учит BERT? | Пробинг attention heads |
| 12 | Практические советы для BERT | Learning rate, batch size, заморозка слоёв |
| 13 | Практические задания | 5 упражнений |

---

---
## Шаг 1. Подготовка окружения

| Библиотека | Зачем |
|-----------|-------|
| **transformers** | Предобученные модели BERT (HuggingFace) |
| **torch** | Фреймворк для обучения нейросетей |
| **matplotlib** | Визуализация: графики, тепловые карты, t-SNE |
| **scikit-learn** | Метрики, t-SNE, confusion matrix |
| **ipywidgets** | Интерактивные виджеты для MLM демо |

In [ ]:
# ============================================================
# ШАГ 1: Устанавливаем и импортируем все нужные библиотеки
# ============================================================

# Устанавливаем трансформеры (если не установлены)
!pip install transformers datasets -q               # тихая установка HuggingFace трансформеров и датасетов

import numpy as np                                  # основная библиотека для массивов и математики
import matplotlib.pyplot as plt                     # для построения графиков и визуализаций
import torch                                        # PyTorch - основной фреймворк
import torch.nn as nn                               # нейросетевые слои
import torch.optim as optim                         # оптимизаторы (AdamW)
from transformers import BertTokenizer, BertModel, BertForMaskedLM, BertForSequenceClassification  # модели BERT
from transformers import get_linear_schedule_with_warmup  # планировщик learning rate
from sklearn.manifold import TSNE                   # t-SNE для визуализации embeddings
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score  # метрики
from sklearn.decomposition import PCA               # PCA для уменьшения размерности
from datasets import load_dataset                   # загрузка датасетов HuggingFace
from tqdm.auto import tqdm                          # прогресс-бар
import ipywidgets as widgets                        # интерактивные виджеты
from ipywidgets import interact, interactive, fixed  # декораторы для интерактива
from IPython.display import display, HTML           # красивый вывод в ноутбуке
import seaborn as sns                               # красивые статистические графики
import warnings                                     # управление предупреждениями

warnings.filterwarnings('ignore')                   # скрываем некритичные предупреждения

# Настройка matplotlib
plt.rcParams['figure.figsize'] = (12, 6)            # размер графиков по умолчанию
plt.rcParams['font.size'] = 12                      # размер шрифта
plt.rcParams['axes.grid'] = True                    # показываем сетку

# Фиксируем random seed для воспроизводимости
torch.manual_seed(42)                               # seed для PyTorch
np.random.seed(42)                                  # seed для NumPy

# Выбираем устройство: GPU если доступен, иначе CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # GPU или CPU

print(f"PyTorch: {torch.__version__}")              # версия PyTorch
print(f"Устройство: {device}")                      # выбранное устройство
print("Все библиотеки импортированы!")              # подтверждение

---
## Шаг 2. Что такое BERT?

**BERT** = **B**idirectional **E**ncoder **R**epresentations from **T**ransformers

До BERT модели для NLP читали текст **в одном направлении** (слева направо).
BERT научился читать текст **в обе стороны одновременно** - как человек!

### Два этапа: предобучение и fine-tuning

**Этап 1: Предобучение (Pre-training)** - на огромном корпусе текста:

| Задача | Описание | Пример |
|--------|----------|--------|
| **MLM** (Masked Language Model) | Предсказываем скрытое слово | "Кот [MASK] на диване" -> "сидит" |
| **NSP** (Next Sentence Prediction) | Идёт ли предложение B после A? | A="Кот спит" B="Он устал" -> Да |

**Этап 2: Fine-tuning (дообучение)** - на конкретной задаче:
- Классификация текста (спам/не спам)
- Вопрос-ответ (QA)
- Распознавание сущностей (NER)
- И многое другое!

```
Предобучение (MLM + NSP на миллиардах слов)
    |
    v
BERT (знания о языке)
    |
    v
Fine-tuning (на вашей задаче) -> Готовая модель
```

> **Ключевая идея:** Вместо обучения с нуля, мы берём модель, которая УЖЕ знает язык,
> и дообучаем её под конкретную задачу. Это как нанять полиглота и научить его юриспруденции,
> а не учить с нуля и язык, и право.

In [ ]:
# ============================================================
# ШАГ 2: Демонстрация идеи transfer learning
# ============================================================

# Сравниваем два подхода: обучение с нуля vs transfer learning
approaches = {                                      # словарь подходов
    "С нуля": {                                     # обучение с нуля
        "Данные": "Нужны миллионы примеров",        # нужно много данных
        "Время": "Дни/недели на GPU",               # долго обучать
        "Качество": "Низкое (мало данных)",          # качество страдает
        "Знания": "Модель учит ВСЁ с нуля",          # никаких предварительных знаний
    },
    "Transfer Learning (BERT)": {                   # transfer learning
        "Данные": "Достаточно тысяч примеров",      # нужно меньше данных
        "Время": "Минуты/часы на GPU",              # быстро дообучать
        "Качество": "Высокое (уже знает язык)",     # высокое качество
        "Знания": "Модель ЗНАЕТ язык из предобучения",  # знания из предобучения
    },
}

# Красивый вывод сравнения
print("=" * 60)
print("Сравнение: Обучение с нуля vs Transfer Learning (BERT)")
print("=" * 60)

for approach, props in approaches.items():          # перебираем подходы
    print(f"\n{approach}:")                          # название подхода
    for key, value in props.items():                # перебираем свойства
        print(f"  {key}: {value}")                  # выводим свойство

print("\n" + "=" * 60)
print("Вывод: Transfer learning быстрее, дешевле и качественнее!")
print("=" * 60)

---
## Шаг 3. Токенизатор BERT: WordPiece

BERT не работает с текстом напрямую. Сначала текст нужно **токенизировать** -
разбить на кусочки (токены), которые модель понимает.

### WordPiece: умное разбиение

| Метод | Пример | Проблема |
|-------|--------|----------|
| По словам | "невероятный" -> ["невероятный"] | Слишком много уникальных слов |
| По символам | "невероятный" -> ["н","е","в","е",...] | Слишком мелко, нет смысла |
| **WordPiece** | "невероятный" -> ["не","##вероят","##ный"] | Баланс: частые слова целиком, редкие - по кускам |

### Специальные токены BERT

| Токен | ID | Зачем |
|-------|-----|-------|
| `[CLS]` | 101 | Маркер начала - его выход используем для классификации |
| `[SEP]` | 102 | Разделитель предложений |
| `[MASK]` | 103 | Замаскированное слово для MLM |
| `[UNK]` | 100 | Неизвестный токен (не в словаре) |
| `[PAD]` | 0 | Заполнение (для одинаковой длины батча) |

In [ ]:
# ============================================================
# ШАГ 3: Загружаем токенизатор BERT и разбираемся в WordPiece
# ============================================================

# Загружаем токенизатор для мультиязычной модели BERT (поддерживает русский)
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')  # мультиязычный BERT

print(f"Размер словаря: {tokenizer.vocab_size}")    # сколько токенов в словаре
print(f"Максимальная длина: {tokenizer.model_max_length}")  # максимальная длина последовательности

# Демонстрируем токенизацию на русском тексте
text = "Нейросети изменяют мир технологий"           # пример текста

# Шаг 1: Токенизация (разбиваем на токены)
tokens = tokenizer.tokenize(text)                    # разбиваем на подтокены WordPiece
print(f"\nТекст: {text}")                            # исходный текст
print(f"Токены: {tokens}")                           # список токенов
print(f"Количество токенов: {len(tokens)}")          # сколько токенов

# Шаг 2: Конвертируем токены в ID (числа)
token_ids = tokenizer.convert_tokens_to_ids(tokens)  # токены -> числа
print(f"ID токенов: {token_ids}")                    # список ID

# Шаг 3: Обратная конвертация (ID -> токены)
decoded_tokens = tokenizer.convert_ids_to_tokens(token_ids)  # числа -> токены
print(f"Обратно: {decoded_tokens}")                  # должны совпасть

# Шаг 4: Полная обработка (как делает модель)
encoded = tokenizer(text, return_tensors='pt')       # полная обработка с тензорами
print(f"\ninput_ids: {encoded['input_ids']}")        # ID с [CLS] и [SEP]
print(f"token_type_ids: {encoded['token_type_ids']}")  # сегменты (0 = первое предложение)
print(f"attention_mask: {encoded['attention_mask']}")  # маска внимания (1 = реальный токен)

In [ ]:
# ============================================================
# ШАГ 3 (продолжение): Демонстрируем ## префикс WordPiece
# ============================================================

# WordPiece разбивает редкие слова на части с префиксом ##
rare_words = [                                       # список редких/длинных слов
    "сверхчувствительность",                          # длинное слово
    "электрокардиограмма",                            # медицинский термин
    "программирование",                               # длинное слово
    "кот",                                            # короткое частое слово
    " transformer ",                                  # английское слово
]

print("WordPiece токенизация редких слов:")
print("=" * 60)

for word in rare_words:                              # перебираем слова
    word = word.strip()                              # убираем пробелы
    tokens = tokenizer.tokenize(word)                # токенизируем
    ids = tokenizer.convert_tokens_to_ids(tokens)    # получаем ID
    has_subwords = any('##' in t for t in tokens)    # есть ли разбиение на подтокены?
    status = "РАЗБИТО" if has_subwords else "ЦЕЛОЕ"  # помечаем статус
    print(f"  {word:30} -> {tokens:30} [{status}]")  # выводим результат

# Сравниваем: как токенизатор обрабатывает предложение целиком
sentence = "Я люблю программировать на Python"        # предложение со смешанными словами
full_tokens = tokenizer.tokenize(sentence)            # токенизируем целиком
print(f"\nПредложение: {sentence}")
print(f"Токены: {full_tokens}")
print(f"\nЗаметка: ## означает 'продолжение слова', а не отдельное слово")

---
## Шаг 4. Архитектура BERT: глубокий разбор

BERT состоит из трёх основных блоков:

```
Входной текст: "Кот сидит"
       |
       v
+------------------+
| 1. Embeddings    |  Token + Position + Segment embeddings
+------------------+
       |
       v
+------------------+
| 2. Encoder x12   |  12 слоёв Transformer Encoder
|   (BERT-base)    |  Каждый: Self-Attention + FFN + LayerNorm
+------------------+
       |
       v
+------------------+
| 3. Выход         |  Скрытые состояния всех токенов
+------------------+
```

### 1. Embeddings (вложения)

Три типа вложений складываются:

| Тип | Что кодирует | Размерность |
|-----|-------------|-------------|
| **Token Embedding** | Какое это слово? | 768 |
| **Position Embedding** | На какой позиции? | 768 |
| **Segment Embedding** | Какому предложению? | 768 |

Итог: **768-мерный вектор** для каждого токена.

### 2. Encoder Layer (один из 12)

```
Вход
 |
 +-> Multi-Head Self-Attention (12 голов) -> Add & Norm
 |
 +-> Feed-Forward Network (768 -> 3072 -> 768) -> Add & Norm
 |
Выход
```

### 3. Размерности BERT-base

| Параметр | Значение |
|----------|----------|
| Слои (L) | 12 |
| Скрытая размерность (H) | 768 |
| Головы внимания (A) | 12 |
| Параметры | ~110M |
| Максимальная длина | 512 токенов |

In [ ]:
# ============================================================
# ШАГ 4: Визуализация архитектуры BERT
# ============================================================

# Рисуем архитектуру BERT в виде схемы
fig, ax = plt.subplots(1, 1, figsize=(14, 10))      # создаём фигуру

# Рисуем схему архитектуры BERT
ax.set_xlim(0, 10)                                  # пределы по X
ax.set_ylim(0, 12)                                  # пределы по Y
ax.axis('off')                                      # убираем оси

# Заголовок
ax.text(5, 11.5, 'Архитектура BERT-base', fontsize=18, ha='center', fontweight='bold')  # заголовок

# Функция для рисования блока
def draw_block(ax, x, y, w, h, text, color, fontsize=10):  # рисуем прямоугольник с текстом
    rect = plt.Rectangle((x, y), w, h, facecolor=color, edgecolor='black', linewidth=2, alpha=0.8)  # прямоугольник
    ax.add_patch(rect)                               # добавляем на холст
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=fontsize, fontweight='bold')  # текст

# Входной текст
draw_block(ax, 3, 10.2, 4, 0.6, 'Вход: "[CLS] Кот сидит [SEP]"', '#FFE0B2', 11)  # вход

# Стрелка вниз
ax.annotate('', xy=(5, 10.2), xytext=(5, 10.0), arrowprops=dict(arrowstyle='->', lw=2))  # стрелка

# Embeddings
draw_block(ax, 1.5, 9.0, 2.2, 0.8, 'Token\nEmbedding', '#BBDEFB', 9)  # токенные вложения
draw_block(ax, 3.9, 9.0, 2.2, 0.8, 'Position\nEmbedding', '#C8E6C9', 9)  # позиционные вложения
draw_block(ax, 6.3, 9.0, 2.2, 0.8, 'Segment\nEmbedding', '#F8BBD0', 9)  # сегментные вложения

# Стрелка к сумме
ax.annotate('', xy=(5, 9.0), xytext=(5, 8.8), arrowprops=dict(arrowstyle='->', lw=2))  # стрелка

# Сумма
draw_block(ax, 3, 8.0, 4, 0.6, 'Sum + LayerNorm', '#E1BEE7', 10)  # сумма + нормализация

# Стрелка вниз
ax.annotate('', xy=(5, 8.0), xytext=(5, 7.8), arrowprops=dict(arrowstyle='->', lw=2))  # стрелка

# Encoder слои
colors_enc = ['#90CAF9', '#81C784', '#FFD54F']      # цвета для слоёв
for i in range(3):                                  # рисуем 3 блока (представляем 12)
    y_pos = 5.8 - i * 1.5                           # позиция по Y
    # Self-Attention
    draw_block(ax, 2, y_pos, 2.5, 0.8, f'Multi-Head\nSelf-Attention', colors_enc[0], 9)  # внимание
    # Add & Norm 1
    draw_block(ax, 4.7, y_pos, 1.5, 0.8, 'Add &\nNorm', '#FFAB91', 8)  # нормализация
    # FFN
    draw_block(ax, 6.4, y_pos, 2.5, 0.8, 'Feed-Forward\n(768->3072->768)', colors_enc[1], 8)  # FFN
    # Add & Norm 2 (под FFN)
    if i < 2:                                       # не рисуем стрелку после последнего
        ax.annotate('', xy=(5, y_pos), xytext=(5, y_pos - 0.2), arrowprops=dict(arrowstyle='->', lw=2))  # стрелка

# Многоточие
ax.text(5, 2.0, '... x12 слоёв ...', ha='center', fontsize=14, fontstyle='italic')  # многоточие

# Выход
draw_block(ax, 3, 0.8, 4, 0.8, 'Выход: скрытые состояния\n(768-мерный вектор на токен)', '#A5D6A7', 10)  # выход

ax.set_title('Архитектура BERT-base (110M параметров)', fontsize=14, pad=20)  # заголовок
plt.tight_layout()                                   # улучшаем компоновку
plt.show()                                           # показываем схему

In [ ]:
# ============================================================
# ШАГ 4 (продолжение): Считаем параметры BERT
# ============================================================

# Давайте посчитаем, откуда берутся 110M параметров
print("Подсчёт параметров BERT-base:")
print("=" * 60)

# Embeddings
vocab_size = 119547                                  # размер словаря мультиязычного BERT
hidden_size = 768                                    # скрытая размерность
max_position = 512                                   # максимальная длина
num_segments = 2                                     # количество сегментов

token_emb = vocab_size * hidden_size                # токенные вложения
position_emb = max_position * hidden_size            # позиционные вложения
segment_emb = num_segments * hidden_size             # сегментные вложения
layer_norm_emb = 2 * hidden_size                     # LayerNorm (gamma + beta)

total_emb = token_emb + position_emb + segment_emb + layer_norm_emb  # всего в embeddings
print(f"Embeddings:")
print(f"  Token:    {vocab_size} x {hidden_size} = {token_emb:>12,}")
print(f"  Position: {max_position} x {hidden_size} = {position_emb:>12,}")
print(f"  Segment:  {num_segments} x {hidden_size} = {segment_emb:>12,}")
print(f"  LayerNorm: 2 x {hidden_size} = {layer_norm_emb:>12,}")
print(f"  ИТОГО Embeddings: {total_emb:>12,}")

# Один Encoder слой
num_heads = 12                                       # количество голов внимания
head_dim = hidden_size // num_heads                  # размерность одной головы
intermediate_size = 3072                             # промежуточный размер FFN

# Self-Attention: Q, K, V проекции + выходная проекция
qkv = 3 * hidden_size * hidden_size                 # Q, K, V матрицы
out_proj = hidden_size * hidden_size                 # выходная проекция
attn_ln = 2 * hidden_size                            # LayerNorm 1

# Feed-Forward Network
ffn_1 = hidden_size * intermediate_size              # первый слой FFN
ffn_2 = intermediate_size * hidden_size              # второй слой FFN
ffn_ln = 2 * hidden_size                             # LayerNorm 2

one_layer = qkv + out_proj + attn_ln + ffn_1 + ffn_2 + ffn_ln  # всего в одном слое
num_layers = 12                                      # количество слоёв
total_encoder = one_layer * num_layers               # всего в энкодере

print(f"\nОдин Encoder слой:")
print(f"  Attention (Q+K+V+Out): {qkv + out_proj:>12,}")
print(f"  LayerNorm 1:           {attn_ln:>12,}")
print(f"  FFN (768->3072->768):  {ffn_1 + ffn_2:>12,}")
print(f"  LayerNorm 2:           {ffn_ln:>12,}")
print(f"  ИТОГО 1 слой:         {one_layer:>12,}")
print(f"\n  12 слоёв:             {total_encoder:>12,}")

total = total_emb + total_encoder                    # общее количество параметров
print(f"\n{'=' * 60}")
print(f"ВСЕГО параметров BERT-base: {total:>12,}")
print(f"(около {total/1e6:.0f}M)")
print(f"{'=' * 60}")

---
## Шаг 5. Загрузка предобученного BERT с HuggingFace

HuggingFace делает загрузку предобученных моделей простой:

```python
from transformers import BertModel
model = BertModel.from_pretrained('bert-base-multilingual-cased')
```

Что происходит под капотом:
1. Скачивается конфиг модели (`config.json`)
2. Скачиваются веса модели (`pytorch_model.bin`)
3. Создаётся архитектура по конфигу
4. Загружаются предобученные веса

Мы используем `bert-base-multilingual-cased` - мультиязычную модель,
которая обучалась на 104 языках, включая русский!

In [ ]:
# ============================================================
# ШАГ 5: Загружаем предобученную модель BERT
# ============================================================

# Загружаем мультиязычную модель BERT (поддерживает русский)
model_name = 'bert-base-multilingual-cased'          # название модели
tokenizer = BertTokenizer.from_pretrained(model_name)  # токенизатор
model = BertModel.from_pretrained(model_name)        # сама модель

# Переводим модель в режим оценки (не обучения)
model.eval()                                         # отключаем dropout и batch norm обучение
model.to(device)                                     # перемещаем на GPU/CPU

# Выводим информацию о модели
print(f"Модель: {model_name}")                       # название модели
print(f"Количество слоёв: {model.config.num_hidden_layers}")  # слои
print(f"Скрытая размерность: {model.config.hidden_size}")     # размерность
print(f"Количество голов внимания: {model.config.num_attention_heads}")  # головы
print(f"Размер словаря: {model.config.vocab_size}")  # размер словаря
print(f"\nМодель загружена и готова к использованию!")

In [ ]:
# ============================================================
# ШАГ 5 (продолжение): Пропускаем текст через BERT
# ============================================================

# Пример: пропускаем русское предложение через BERT
text = "Машинное обучение трансформирует индустрию"  # пример текста

# Шаг 1: Токенизация
inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)  # токенизируем
inputs = {k: v.to(device) for k, v in inputs.items()}  # перемещаем на устройство

print("Входные тензоры:")
print(f"  input_ids shape: {inputs['input_ids'].shape}")     # форма ID
print(f"  attention_mask shape: {inputs['attention_mask'].shape}")  # форма маски

# Шаг 2: Прямой проход (forward pass)
with torch.no_grad():                                # не вычисляем градиенты (экономим память)
    outputs = model(**inputs)                        # пропускаем через модель

# Шаг 3: Изучаем выходы
last_hidden_state = outputs.last_hidden_state        # скрытые состояния последнего слоя
pooler_output = outputs.pooler_output                # выход [CLS] токена через tanh

print(f"\nВыходы BERT:")
print(f"  last_hidden_state shape: {last_hidden_state.shape}")  # [1, num_tokens, 768]
print(f"  pooler_output shape: {pooler_output.shape}")          # [1, 768]

# Декодируем токены для наглядности
token_ids = inputs['input_ids'][0].cpu().numpy()     # получаем ID токенов
tokens = tokenizer.convert_ids_to_tokens(token_ids)  # конвертируем обратно в текст
print(f"\nТокены: {tokens}")
print(f"Каждый токен -> 768-мерный вектор")

---
## Шаг 6. BERT embeddings: исследование представления

Каждый токен на выходе BERT - это 768-мерный вектор.
Эти векторы **кодируют смысл** слов в контексте.

Но как их визуализировать? С помощью **t-SNE** - алгоритма,
который "сплющивает" 768 измерений в 2D, сохраняя относительные расстояния.

> Слова с похожим смыслом должны быть рядом на графике!

In [ ]:
# ============================================================
# ШАГ 6: Визуализация BERT embeddings через t-SNE
# ============================================================

# Набор слов для визуализации (разные категории)
words = [                                            # список слов разных категорий
    # Животные
    "кот", "собака", "птица", "рыба", "лошадь",
    # Еда
    "хлеб", "молоко", "мясо", "рис", "яблоко",
    # Технологии
    "компьютер", "телефон", "интернет", "программа", "сервер",
    # Эмоции
    "радость", "грусть", "страх", "любовь", "злость",
]

# Получаем embeddings для каждого слова
embeddings = []                                      # список вложений
labels = []                                          # метки категорий
categories = ["Животные"] * 5 + ["Еда"] * 5 + ["Технологии"] * 5 + ["Эмоции"] * 5  # категории

for word in words:                                   # перебираем слова
    inp = tokenizer(word, return_tensors='pt').to(device)  # токенизируем
    with torch.no_grad():                            # без градиентов
        out = model(**inp)                           # прямой проход
    # Берём вектор [CLS] токена как представление слова
    cls_embedding = out.pooler_output.cpu().numpy()[0]  # [CLS] вложение
    embeddings.append(cls_embedding)                 # добавляем в список
    labels.append(word)                              # добавляем метку

embeddings = np.array(embeddings)                    # преобразуем в numpy массив
print(f"Shape embeddings: {embeddings.shape}")       # форма массива

# Сначала PCA для ускорения t-SNE (768 -> 50)
pca = PCA(n_components=50)                           # PCA до 50 компонент
embeddings_pca = pca.fit_transform(embeddings)       # применяем PCA
print(f"Explained variance ratio (PCA): {pca.explained_variance_ratio_.sum():.3f}")  # доля дисперсии

# t-SNE: 50 -> 2
tsne = TSNE(n_components=2, random_state=42, perplexity=5, n_iter=1000)  # t-SNE
embeddings_2d = tsne.fit_transform(embeddings_pca)   # применяем t-SNE

# Визуализация
fig, ax = plt.subplots(figsize=(14, 10))             # создаём фигуру

colors_map = {                                       # цвета для категорий
    "Животные": "#e74c3c",                           # красный
    "Еда": "#2ecc71",                                # зелёный
    "Технологии": "#3498db",                         # синий
    "Эмоции": "#f39c12",                             # оранжевый
}

for cat in colors_map:                               # перебираем категории
    mask = [c == cat for c in categories]            # маска для категории
    idx = [i for i, m in enumerate(mask) if m]       # индексы
    ax.scatter(                                      # рисуем точки
        embeddings_2d[idx, 0],                       # координаты X
        embeddings_2d[idx, 1],                       # координаты Y
        c=colors_map[cat],                           # цвет
        s=200,                                       # размер точек
        alpha=0.8,                                   # прозрачность
        label=cat,                                   # метка для легенды
        edgecolors='black',                          # обводка
        linewidths=1                                 # толщина обводки
    )
    # Подписываем слова
    for i in idx:                                    # перебираем индексы
        ax.annotate(                                 # подпись
            labels[i],                               # текст
            (embeddings_2d[i, 0], embeddings_2d[i, 1]),  # позиция
            fontsize=11,                             # размер шрифта
            fontweight='bold',                       # жирный
            xytext=(5, 5),                           # смещение
            textcoords='offset points'               # тип смещения
        )

ax.set_title('BERT embeddings: визуализация t-SNE\nСлова с похожим смыслом группируются', fontsize=16)  # заголовок
ax.legend(fontsize=12, loc='best')                   # легенда
ax.set_xlabel('t-SNE компонента 1', fontsize=12)    # ось X
ax.set_ylabel('t-SNE компонента 2', fontsize=12)    # ось Y
plt.tight_layout()                                   # улучшаем компоновку
plt.show()                                           # показываем график

In [ ]:
# ============================================================
# ШАГ 6 (продолжение): Контекстные embeddings - одно слово, разные контексты
# ============================================================

# BERT даёт РАЗНЫЕ вложения для одного слова в разных контекстах!
# Классический пример: "банк" может означать финансовое учреждение или берег реки

sentences = [                                        # предложения с разными значениями слова "банк"
    "Я положил деньги в банк",                       # финансовый контекст
    "Мы сидели на берегу реки у крутого банка",      # речной контекст
    "Банк России повысил ключевую ставку",           # финансовый контекст
    "Рыбаки стояли на берегу у песчаного банка",     # речной контекст
    "Этот банк предлагает выгодный кредит",          # финансовый контекст
    "Зелёный банка реки зарос камышом",              # речной контекст
]

# Получаем embedding токена "банк" из каждого предложения
bank_embeddings = []                                 # список вложений "банк"
context_labels = []                                  # метки контекста

for sent in sentences:                               # перебираем предложения
    inputs = tokenizer(sent, return_tensors='pt').to(device)  # токенизируем
    with torch.no_grad():                            # без градиентов
        outputs = model(**inputs)                    # прямой проход

    # Находим позицию токена "банк"
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])  # получаем токены
    bank_idx = None                                  # индекс токена "банк"
    for i, token in enumerate(tokens):               # перебираем токены
        if 'банк' in token.lower():                  # нашли "банк"
            bank_idx = i                             # запоминаем индекс
            break                                    # выходим из цикла

    if bank_idx is not None:                         # если нашли
        # Берём скрытое состояние этого токена
        emb = outputs.last_hidden_state[0, bank_idx].cpu().numpy()  # вложение токена
        bank_embeddings.append(emb)                  # добавляем
        # Определяем контекст
        is_financial = any(w in sent for w in ["деньги", "ставку", "кредит", "России"])  # финансовый?
        context_labels.append("Финансы" if is_financial else "Река/берег")  # метка

bank_embeddings = np.array(bank_embeddings)          # numpy массив

# PCA + t-SNE для визуализации
if len(bank_embeddings) >= 5:                        # если достаточно примеров
    pca_bank = PCA(n_components=min(4, len(bank_embeddings) - 1))  # PCA
    emb_pca = pca_bank.fit_transform(bank_embeddings)  # применяем PCA
    tsne_bank = TSNE(n_components=2, random_state=42, perplexity=min(3, len(bank_embeddings)-1))  # t-SNE
    emb_2d = tsne_bank.fit_transform(emb_pca)        # применяем t-SNE

    # Рисуем
    fig, ax = plt.subplots(figsize=(10, 8))          # создаём фигуру
    colors_ctx = {"Финансы": "#e74c3c", "Река/берег": "#3498db"}  # цвета

    for ctx in colors_ctx:                           # перебираем контексты
        mask = [l == ctx for l in context_labels]    # маска
        idx = [i for i, m in enumerate(mask) if m]   # индексы
        ax.scatter(emb_2d[idx, 0], emb_2d[idx, 1],  # точки
                   c=colors_ctx[ctx], s=300, alpha=0.8,  # параметры
                   label=ctx, edgecolors='black', linewidths=2)  # стили
        for i in idx:                                # подписываем
            ax.annotate(sentences[i][:30] + "...",   # сокращённое предложение
                        (emb_2d[i, 0], emb_2d[i, 1]),  # позиция
                        fontsize=9, xytext=(10, 5),  # смещение
                        textcoords='offset points')   # тип смещения

    ax.set_title('Контекстные embeddings: слово "банк" в разных значениях\nBERT различает смысл по контексту!', fontsize=14)  # заголовок
    ax.legend(fontsize=12)                           # легенда
    plt.tight_layout()                               # компоновка
    plt.show()                                       # показываем

print("\nКлючевой вывод: BERT даёт РАЗНЫЕ векторы для одного слова в разных контекстах!")
print("Это главное преимущество перед статическими вложениями (Word2Vec, GloVe).")

---
## Шаг 7. Masked Language Model: предсказание [MASK]

Одна из задач предобучения BERT - **Masked Language Model (MLM)**:
модель учится предсказывать замаскированные слова.

Принцип:
1. Берём предложение: "Кот сидит на диване"
2. Заменяем 15% слов на `[MASK]`: "Кот [MASK] на диване"
3. Модель предсказывает: "сидит" (вероятность 0.85)

> Это как "Виселица" или заполнение пропусков в тесте!

In [ ]:
# ============================================================
# ШАГ 7: Загружаем модель для Masked Language Model
# ============================================================

# Загружаем специальную версию BERT для MLM
mlm_model = BertForMaskedLM.from_pretrained(model_name)  # модель для MLM
mlm_model.eval()                                     # режим оценки
mlm_model.to(device)                                 # перемещаем на устройство

print(f"MLM модель загружена: {model_name}")         # подтверждение
print(f"Количество параметров: {sum(p.numel() for p in mlm_model.parameters()):,}")  # параметры

In [ ]:
# ============================================================
# ШАГ 7 (продолжение): Функция для предсказания замаскированных слов
# ============================================================

def predict_mask(text, top_k=5):                     # функция предсказания [MASK]
    # Предсказываем top-k слов для позиции [MASK]
    inputs = tokenizer(text, return_tensors='pt').to(device)  # токенизируем
    mask_token_id = tokenizer.mask_token_id          # ID токена [MASK]

    # Находим позицию [MASK]
    mask_idx = (inputs['input_ids'] == mask_token_id).nonzero()  # ищем [MASK]
    if len(mask_idx) == 0:                           # если не нашли
        return "Нет токена [MASK] в тексте!"         # ошибка
    mask_pos = mask_idx[0][1].item()                 # позиция [MASK]

    # Прямой проход
    with torch.no_grad():                            # без градиентов
        outputs = mlm_model(**inputs)                # предсказание

    # Получаем logits для позиции [MASK]
    mask_logits = outputs.logits[0, mask_pos]        # logits для [MASK]
    probs = torch.softmax(mask_logits, dim=-1)       # вероятности

    # Топ-k предсказаний
    top_k_probs, top_k_ids = torch.topk(probs, top_k)  # топ-k

    results = []                                     # список результатов
    for prob, token_id in zip(top_k_probs, top_k_ids):  # перебираем
        token = tokenizer.convert_ids_to_tokens([token_id])[0]  # декодируем токен
        results.append((token, prob.item()))         # добавляем в список

    return results                                   # возвращаем результаты

# Демонстрация
examples = [                                         # примеры для MLM
    "Искусственный [MASK] изменяет мир",             # интеллект
    "Кот [MASK] на диване",                          # сидит
    "Студенты [MASK] в университете",                # учатся
    "Машинное [MASK] - это будущее",                 # обучение
]

for text in examples:                                # перебираем примеры
    results = predict_mask(text, top_k=5)            # предсказываем
    print(f"\nТекст: {text}")
    print(f"Топ-5 предсказаний:")
    for token, prob in results:                      # перебираем результаты
        bar = '#' * int(prob * 50)                   # визуальная полоска
        print(f"  {token:15} {prob:.4f} {bar}")     # выводим

In [ ]:
# ============================================================
# ШАГ 7 (продолжение): Интерактивный MLM демо с ipywidgets
# ============================================================

# Создаём интерактивный виджет для MLM
text_input = widgets.Text(                           # текстовое поле
    value='Машинное [MASK] изменяет мир',            # значение по умолчанию
    description='Текст с [MASK]:',                   # подпись
    layout=widgets.Layout(width='600px')             # ширина
)

top_k_slider = widgets.IntSlider(                    # слайдер для top-k
    value=10,                                        # значение по умолчанию
    min=3,                                           # минимум
    max=20,                                          # максимум
    step=1,                                          # шаг
    description='Top-K:',                            # подпись
)

output_area = widgets.Output()                       # область вывода

def on_change(change):                               # обработчик изменений
    with output_area:                                # в области вывода
        output_area.clear_output()                   # очищаем
        text = text_input.value                      # получаем текст
        if '[MASK]' not in text:                     # проверяем наличие [MASK]
            print("Добавьте [MASK] в текст!")        # предупреждение
            return                                   # выходим
        results = predict_mask(text, top_k_slider.value)  # предсказываем
        if isinstance(results, str):                 # если ошибка
            print(results)                           # выводим ошибку
            return                                   # выходим
        print(f"Текст: {text}\n")
        print(f"Топ-{top_k_slider.value} предсказаний:")

        # Визуализация
        tokens = [r[0] for r in results]             # токены
        probs = [r[1] for r in results]              # вероятности

        fig, ax = plt.subplots(figsize=(10, 5))      # фигура
        bars = ax.barh(range(len(tokens)), probs, color='steelblue')  # горизонтальные столбцы
        ax.set_yticks(range(len(tokens)))            # метки по Y
        ax.set_yticklabels(tokens)                   # подписи
        ax.set_xlabel('Вероятность')                 # ось X
        ax.set_title(f'Предсказания для [MASK]: {text}')  # заголовок
        ax.invert_yaxis()                            # инвертируем ось

        for bar, prob in zip(bars, probs):           # добавляем значения
            ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,  # позиция
                    f'{prob:.4f}', va='center', fontsize=10)  # текст

        plt.tight_layout()                           # компоновка
        plt.show()                                   # показываем

text_input.observe(on_change, names='value')         # наблюдаем за текстом
top_k_slider.observe(on_change, names='value')       # наблюдаем за слайдером

# Запускаем первый раз
on_change(None)                                      # инициализация

# Показываем виджеты
display(widgets.VBox([text_input, top_k_slider, output_area]))  # отображаем

---
## Шаг 8. Fine-tuning BERT для классификации

Теперь самое интересное - **дообучим** BERT для классификации текста!

Мы будем работать с отзывами: **положительные** vs **отрицательные**.

### Архитектура для классификации

```
BERT (12 слоёв, заморожены или нет)
  |
  v
[CLS] токен -> 768-мерный вектор
  |
  v
Dropout (0.1)
  |
  v
Linear (768 -> 2)  [положительный / отрицательный]
  |
  v
Softmax -> вероятности классов
```

### План обучения

1. Подготовим датасет (маленький, для скорости)
2. Создадим DataLoader
3. Настроим оптимизатор (AdamW) и планировщик
4. Запустим цикл обучения
5. Сохраним лучшую модель

In [ ]:
# ============================================================
# ШАГ 8: Создаём датасет для классификации
# ============================================================

# Создаём синтетический датасет отзывов на русском
positive_reviews = [                                 # положительные отзывы
    "Отличный фильм, очень понравился!",
    "Прекрасная книга, рекомендую всем",
    "Замечательный сервис, быстро и качественно",
    "Великолепная работа, превосходный результат",
    "Хорошая погода сегодня, настроение прекрасное",
    "Мне очень нравится этот ресторан",
    "Потрясающий концерт, невероятные эмоции",
    "Прекрасный отель, уютные номера",
    "Отличный преподаватель, понятно объясняет",
    "Замечательный подарок, я очень рад",
    "Восхитительная музыка, слушаю каждый день",
    "Прекрасный день, всё идёт по плану",
    "Классный смартфон, работает быстро",
    "Отличная команда, профессионалы своего дела",
    "Чудесный город, красивая архитектура",
    "Великолепная еда, шеф-повар мастер",
    "Хороший фильм с глубоким смыслом",
    "Приятный персонал, вежливое обслуживание",
    "Прекрасная погода для прогулки",
    "Замечательный проект, интересная работа",
    "Отличный выбор, я доволен покупкой",
    "Восхитительный вид из окна",
    "Хорошая новость, обрадовался",
    "Прекрасный результат превзошёл ожидания",
    "Классная вечеринка, отлично провели время",
    "Великолепная организация мероприятия",
    "Замечательный сосед, тихий и дружелюбный",
    "Хороший врач, помог вылечиться",
    "Прекрасная возможность для развития",
    "Отличная идея, давайте реализуем",
]

negative_reviews = [                                 # отрицательные отзывы
    "Ужасный фильм, потратил время зря",
    "Плохая книга, скучно и неинтересно",
    "Страшный сервис, долго и некачественно",
    "Разочаровала работа, результат ужасный",
    "Дрянная погода, настроение отвратительное",
    "Не нравится этот ресторан, грязно",
    "Скучный концерт, зря пошёл",
    "Ужасный отель, грязные номера",
    "Плохой преподаватель, ничего не понятно",
    "Разочаровал подарок, не ожидал такого",
    "Невыносимая музыка, болят уши",
    "Плохой день, всё пошло не так",
    "Ужасный смартфон, постоянно зависает",
    "Плохая команда, не умеют работать",
    "Скучный город, смотреть нечего",
    "Отвратительная еда, есть невозможно",
    "Плохой фильм, сюжет предсказуем",
    "Грубый персонал, хамское обслуживание",
    "Ужасная погода, идти невозможно",
    "Провальный проект, зря потрачены силы",
    "Плохая покупка, жаль потраченных денег",
    "Унылый вид из окна, ничего красивого",
    "Плохая новость, огорчился",
    "Разочаровывающий результат, не стоило ждать",
    "Скучная вечеринка, рано ушёл",
    "Плохая организация, полный хаос",
    "Шумный сосед, спать невозможно",
    "Некомпетентный врач, не помог",
    "Упущенная возможность, обидно",
    "Плохая идея, ничего не вышло",
]

# Объединяем и создаём метки
texts = positive_reviews + negative_reviews           # все тексты
labels = [1] * len(positive_reviews) + [0] * len(negative_reviews)  # 1 = позитив, 0 = негатив

print(f"Всего примеров: {len(texts)}")               # количество примеров
print(f"Положительных: {sum(labels)}")               # количество позитивных
print(f"Отрицательных: {len(labels) - sum(labels)}")  # количество негативных
print(f"\nПример положительного: {positive_reviews[0]}")
print(f"Пример отрицательного: {negative_reviews[0]}")

In [ ]:
# ============================================================
# ШАГ 8 (продолжение): Создаём DataLoader
# ============================================================

from torch.utils.data import Dataset, DataLoader, random_split  # утилиты для данных

# Создаём класс датасета
class ReviewDataset(Dataset):                        # класс датасета отзывов
    def __init__(self, texts, labels, tokenizer, max_len=64):  # инициализация
        self.texts = texts                           # тексты
        self.labels = labels                         # метки
        self.tokenizer = tokenizer                   # токенизатор
        self.max_len = max_len                       # максимальная длина

    def __len__(self):                               # длина датасета
        return len(self.texts)                       # количество примеров

    def __getitem__(self, idx):                      # получение примера по индексу
        text = str(self.texts[idx])                  # текст
        label = self.labels[idx]                     # метка

        # Токенизируем текст
        encoding = self.tokenizer(                   # токенизация
            text,                                    # текст
            max_length=self.max_len,                 # максимальная длина
            padding='max_length',                    # заполнение до max_len
            truncation=True,                         # обрезка если длиннее
            return_tensors='pt'                      # вернуть тензоры
        )

        return {                                     # возвращаем словарь
            'input_ids': encoding['input_ids'].flatten(),  # ID токенов
            'attention_mask': encoding['attention_mask'].flatten(),  # маска внимания
            'token_type_ids': encoding['token_type_ids'].flatten(),  # типы токенов
            'labels': torch.tensor(label, dtype=torch.long),  # метка
        }

# Создаём датасет
dataset = ReviewDataset(texts, labels, tokenizer, max_len=64)  # датасет

# Разбиваем на обучающую и валидационную выборки (80/20)
train_size = int(0.8 * len(dataset))                 # 80% для обучения
val_size = len(dataset) - train_size                 # 20% для валидации
train_dataset, val_dataset = random_split(dataset, [train_size, val_size],  # разбиваем
                                          generator=torch.Generator().manual_seed(42))  # фиксируем seed

# Создаём DataLoader-ы
batch_size = 8                                       # размер батча
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)  # обучающий
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)    # валидационный

print(f"Обучающая выборка: {len(train_dataset)} примеров")   # размер обучающей
print(f"Валидационная выборка: {len(val_dataset)} примеров")  # размер валидационной
print(f"Батчей для обучения: {len(train_loader)}")    # количество батчей
print(f"Батчей для валидации: {len(val_loader)}")    # количество батчей

In [ ]:
# ============================================================
# ШАГ 8 (продолжение): Создаём модель для классификации
# ============================================================

# Загружаем модель BERT для классификации последовательностей
cls_model = BertForSequenceClassification.from_pretrained(  # модель классификации
    model_name,                                      # название модели
    num_labels=2,                                    # 2 класса: позитив/негатив
    problem_type="single_label_classification"       # тип задачи
)
cls_model.to(device)                                 # перемещаем на устройство

# Настраиваем параметры обучения
epochs = 5                                           # количество эпох
learning_rate = 2e-5                                 # learning rate (стандарт для BERT)

# Оптимизатор AdamW (Adam с weight decay)
optimizer = optim.AdamW(                             # оптимизатор
    cls_model.parameters(),                          # параметры модели
    lr=learning_rate,                                # learning rate
    weight_decay=0.01                                # weight decay (L2 регуляризация)
)

# Общее количество шагов обучения
total_steps = len(train_loader) * epochs             # общее количество шагов

# Планировщик learning rate с разогревом
scheduler = get_linear_schedule_with_warmup(          # планировщик
    optimizer,                                       # оптимизатор
    num_warmup_steps=int(0.1 * total_steps),         # 10% шагов для разогрева
    num_training_steps=total_steps                   # общее количество шагов
)

print(f"Модель для классификации создана!")
print(f"Параметры: {sum(p.numel() for p in cls_model.parameters()):,}")
print(f"Обучаемые: {sum(p.numel() for p in cls_model.parameters() if p.requires_grad):,}")
print(f"\nЭпохи: {epochs}")
print(f"Learning rate: {learning_rate}")
print(f"Всего шагов: {total_steps}")
print(f"Шагов разогрева: {int(0.1 * total_steps)}")

In [ ]:
# ============================================================
# ШАГ 8 (продолжение): Цикл обучения (fine-tuning)
# ============================================================

# Списки для хранения метрик обучения
train_losses = []                                    # потери на обучении
val_losses = []                                      # потери на валидации
train_accs = []                                      # точность на обучении
val_accs = []                                        # точность на валидации

best_val_acc = 0.0                                   # лучшая точность на валидации

for epoch in range(epochs):                          # перебираем эпохи
    # === Обучение ===
    cls_model.train()                                # режим обучения
    total_train_loss = 0                             # суммарная потеря
    correct_train = 0                                # правильные предсказания
    total_train = 0                                  # всего примеров

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):  # перебираем батчи
        # Перемещаем данные на устройство
        input_ids = batch['input_ids'].to(device)    # ID токенов
        attention_mask = batch['attention_mask'].to(device)  # маска внимания
        token_type_ids = batch['token_type_ids'].to(device)  # типы токенов
        labels_batch = batch['labels'].to(device)    # метки

        # Обнуляем градиенты
        optimizer.zero_grad()                        # обнуляем градиенты

        # Прямой проход
        outputs = cls_model(                         # прямой проход
            input_ids=input_ids,                     # ID токенов
            attention_mask=attention_mask,           # маска внимания
            token_type_ids=token_type_ids,           # типы токенов
            labels=labels_batch                      # метки (для вычисления loss)
        )

        loss = outputs.loss                          # потеря
        logits = outputs.logits                      # logits

        # Обратный проход
        loss.backward()                              # вычисляем градиенты
        torch.nn.utils.clip_grad_norm_(cls_model.parameters(), 1.0)  # обрезаем градиенты
        optimizer.step()                             # обновляем веса
        scheduler.step()                             # обновляем learning rate

        # Считаем метрики
        total_train_loss += loss.item()              # добавляем потерю
        preds = torch.argmax(logits, dim=-1)         # предсказания
        correct_train += (preds == labels_batch).sum().item()  # правильные
        total_train += labels_batch.size(0)          # всего

    # Средние метрики за эпоху
    avg_train_loss = total_train_loss / len(train_loader)  # средняя потеря
    train_acc = correct_train / total_train          # точность

    # === Валидация ===
    cls_model.eval()                                 # режим оценки
    total_val_loss = 0                               # суммарная потеря
    correct_val = 0                                  # правильные предсказания
    total_val = 0                                    # всего примеров

    with torch.no_grad():                            # без градиентов
        for batch in val_loader:                     # перебираем батчи
            input_ids = batch['input_ids'].to(device)  # ID токенов
            attention_mask = batch['attention_mask'].to(device)  # маска
            token_type_ids = batch['token_type_ids'].to(device)  # типы
            labels_batch = batch['labels'].to(device)  # метки

            outputs = cls_model(                     # прямой проход
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                labels=labels_batch
            )

            total_val_loss += outputs.loss.item()    # добавляем потерю
            preds = torch.argmax(outputs.logits, dim=-1)  # предсказания
            correct_val += (preds == labels_batch).sum().item()  # правильные
            total_val += labels_batch.size(0)        # всего

    avg_val_loss = total_val_loss / len(val_loader)  # средняя потеря
    val_acc = correct_val / total_val                # точность

    # Сохраняем метрики
    train_losses.append(avg_train_loss)              # потеря обучения
    val_losses.append(avg_val_loss)                  # потеря валидации
    train_accs.append(train_acc)                     # точность обучения
    val_accs.append(val_acc)                         # точность валидации

    # Сохраняем лучшую модель
    if val_acc > best_val_acc:                       # если лучше
        best_val_acc = val_acc                       # обновляем лучшую

    print(f"Epoch {epoch+1}/{epochs}: "
          f"Train Loss={avg_train_loss:.4f}, Train Acc={train_acc:.4f}, "
          f"Val Loss={avg_val_loss:.4f}, Val Acc={val_acc:.4f}")

print(f"\nЛучшая точность на валидации: {best_val_acc:.4f}")

---
## Шаг 9. Оценка дообученного BERT

Теперь оценим нашу модель подробнее:
- Кривые обучения (loss и accuracy)
- Classification report (precision, recall, f1)
- Confusion matrix (матрица ошибок)

In [ ]:
# ============================================================
# ШАГ 9: Визуализация кривых обучения
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))     # создаём фигуру с двумя подграфиками

# График потерь (Loss)
axes[0].plot(range(1, epochs+1), train_losses, 'b-o', label='Train Loss', linewidth=2, markersize=8)  # обучение
axes[0].plot(range(1, epochs+1), val_losses, 'r-o', label='Val Loss', linewidth=2, markersize=8)      # валидация
axes[0].set_xlabel('Эпоха', fontsize=14)             # ось X
axes[0].set_ylabel('Loss', fontsize=14)              # ось Y
axes[0].set_title('Кривая потерь (Loss)', fontsize=16)  # заголовок
axes[0].legend(fontsize=12)                          # легенда
axes[0].set_xticks(range(1, epochs+1))              # метки по X

# График точности (Accuracy)
axes[1].plot(range(1, epochs+1), train_accs, 'b-o', label='Train Accuracy', linewidth=2, markersize=8)  # обучение
axes[1].plot(range(1, epochs+1), val_accs, 'r-o', label='Val Accuracy', linewidth=2, markersize=8)      # валидация
axes[1].set_xlabel('Эпоха', fontsize=14)             # ось X
axes[1].set_ylabel('Accuracy', fontsize=14)          # ось Y
axes[1].set_title('Кривая точности (Accuracy)', fontsize=16)  # заголовок
axes[1].legend(fontsize=12)                          # легенда
axes[1].set_xticks(range(1, epochs+1))              # метки по X
axes[1].set_ylim(0.4, 1.05)                         # пределы по Y

plt.suptitle('Кривые обучения Fine-tuned BERT', fontsize=18, y=1.02)  # общий заголовок
plt.tight_layout()                                   # компоновка
plt.show()                                           # показываем

In [ ]:
# ============================================================
# ШАГ 9 (продолжение): Подробная оценка на валидации
# ============================================================

# Собираем все предсказания и истинные метки
cls_model.eval()                                     # режим оценки
all_preds = []                                       # все предсказания
all_labels = []                                      # все истинные метки
all_probs = []                                       # все вероятности

with torch.no_grad():                                # без градиентов
    for batch in val_loader:                         # перебираем батчи
        input_ids = batch['input_ids'].to(device)    # ID токенов
        attention_mask = batch['attention_mask'].to(device)  # маска
        token_type_ids = batch['token_type_ids'].to(device)  # типы
        labels_batch = batch['labels'].to(device)    # метки

        outputs = cls_model(                         # прямой проход
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        probs = torch.softmax(outputs.logits, dim=-1)  # вероятности
        preds = torch.argmax(probs, dim=-1)          # предсказания

        all_preds.extend(preds.cpu().numpy())        # добавляем предсказания
        all_labels.extend(labels_batch.cpu().numpy())  # добавляем метки
        all_probs.extend(probs.cpu().numpy())        # добавляем вероятности

# Classification report
class_names = ['Негативный', 'Позитивный']           # имена классов
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))  # отчёт

# Accuracy
acc = accuracy_score(all_labels, all_preds)          # точность
print(f"Accuracy: {acc:.4f}")                        # выводим

In [ ]:
# ============================================================
# ШАГ 9 (продолжение): Confusion Matrix
# ============================================================

# Строим confusion matrix
cm = confusion_matrix(all_labels, all_preds)         # матрица ошибок

fig, ax = plt.subplots(figsize=(8, 6))               # создаём фигуру

# Визуализируем с помощью seaborn
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',   # тепловая карта
            xticklabels=class_names,                  # метки по X
            yticklabels=class_names,                  # метки по Y
            ax=ax,                                   # оси
            annot_kws={'size': 16})                   # размер шрифта аннотаций

ax.set_xlabel('Предсказанный класс', fontsize=14)    # ось X
ax.set_ylabel('Истинный класс', fontsize=14)         # ось Y
ax.set_title('Confusion Matrix: BERT классификация отзывов', fontsize=16)  # заголовок

plt.tight_layout()                                   # компоновка
plt.show()                                           # показываем

# Примеры предсказаний
print("\nПримеры предсказаний:")
print("=" * 60)
cls_model.eval()                                     # режим оценки
test_texts = [                                       # тестовые тексты
    "Этот фильм просто замечательный!",              # позитив
    "Ужасный сервис, никогда не вернусь",            # негатив
    "Отличная книга, всем советую",                  # позитив
    "Потратил деньги зря, разочарован",              # негатив
]

for text in test_texts:                              # перебираем тесты
    inputs = tokenizer(text, return_tensors='pt', max_length=64,  # токенизируем
                       padding='max_length', truncation=True).to(device)  # на устройство
    with torch.no_grad():                            # без градиентов
        outputs = cls_model(**inputs)                # предсказание
    prob = torch.softmax(outputs.logits, dim=-1)     # вероятности
    pred = torch.argmax(prob, dim=-1).item()         # класс
    confidence = prob[0][pred].item()                # уверенность
    label = class_names[pred]                        # имя класса
    print(f"  Текст: {text}")
    print(f"  Предсказание: {label} (уверенность: {confidence:.4f})")
    print()

---
## Шаг 10. BERT vs предыдущие подходы

Как BERT соотносится с более ранними методами NLP?

| Метод | Год | Направление | Предобучение | Контекстность |
|-------|-----|-------------|-------------|---------------|
| Word2Vec | 2013 | Нет | Нет (на одном корпусе) | Нет (статический) |
| GloVe | 2014 | Нет | Нет | Нет (статический) |
| ELMo | 2018 | Двунаправленный | Да (LM) | Да (через LSTM) |
| ULMFiT | 2018 | Прямой | Да (LM) | Частично |
| OpenAI GPT | 2018 | Прямой | Да (LM) | Да (через Transformer) |
| **BERT** | **2018** | **Двунаправленный** | **Да (MLM+NSP)** | **Да (через Transformer)** |

### Ключевые преимущества BERT

1. **Двунаправленность**: видит контекст С ОБЕИХ сторон
2. **MLM**: более сложная задача предобучения -> лучше учит
3. **Transfer Learning**: одна предобученная модель -> много задач
4. **Масштабируемость**: BERT-large (24 слоя, 340M параметров) ещё лучше

In [ ]:
# ============================================================
# ШАГ 10: Визуализация сравнения подходов
# ============================================================

# Данные для сравнения
methods = ['Word2Vec', 'ELMo', 'OpenAI GPT', 'BERT-base', 'BERT-large']  # методы
years = [2013, 2018, 2018, 2018, 2018]              # годы выхода
params_m = [0, 94, 110, 110, 340]                   # параметры (млн)
glue_scores = [0, 0, 72.0, 79.6, 82.1]              # GLUE score
contextual = [0, 1, 1, 2, 2]                        # контекстность: 0=нет, 1=однонапр, 2=двунапр

fig, axes = plt.subplots(1, 3, figsize=(18, 6))     # фигура с 3 подграфиками

# График 1: Количество параметров
colors_params = ['#95a5a6', '#3498db', '#2ecc71', '#e74c3c', '#c0392b']  # цвета
bars = axes[0].bar(methods, params_m, color=colors_params, edgecolor='black')  # столбцы
axes[0].set_ylabel('Параметры (млн)', fontsize=12)  # ось Y
axes[0].set_title('Количество параметров', fontsize=14)  # заголовок
for bar, val in zip(bars, params_m):                 # подписываем столбцы
    if val > 0:                                     # если значение > 0
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,  # позиция
                     f'{val}M', ha='center', fontsize=10, fontweight='bold')  # текст

# График 2: GLUE Score
bars2 = axes[1].bar(methods, glue_scores, color=colors_params, edgecolor='black')  # столбцы
axes[1].set_ylabel('GLUE Score', fontsize=12)       # ось Y
axes[1].set_title('Качество (GLUE benchmark)', fontsize=14)  # заголовок
for bar, val in zip(bars2, glue_scores):             # подписываем
    if val > 0:                                     # если значение > 0
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,  # позиция
                     f'{val:.1f}', ha='center', fontsize=10, fontweight='bold')  # текст

# График 3: Направление контекста
context_labels = ['Статический', 'Однонаправленный', 'Двунаправленный']  # метки
context_counts = [1, 2, 2]                           # количество методов
context_colors = ['#95a5a6', '#3498db', '#e74c3c']   # цвета
axes[2].bar(context_labels, context_counts, color=context_colors, edgecolor='black')  # столбцы
axes[2].set_ylabel('Количество методов', fontsize=12)  # ось Y
axes[2].set_title('Тип контекстности', fontsize=14)  # заголовок

plt.suptitle('Сравнение подходов к представлению текста', fontsize=18, y=1.02)  # общий заголовок
plt.tight_layout()                                   # компоновка
plt.show()                                           # показываем

print("Вывод: BERT - самый качественный подход благодаря двунаправленности и MLM!")

---
## Шаг 11. Что учит BERT? Пробинг attention heads

BERT имеет 12 слоёв x 12 голов внимания = **144 внимания**.
Что они изучают?

Исследования показывают, что разные головы внимания специализируются:
- **Синтаксис**: какая словоформа (падеж, число)
- **Синтаксис**: кто подчиняет кого (подлежащее -> сказуемое)
- **Кореферентность**: кто к кому относится (местоимение -> существительное)
- **Семантика**: смысловая близость слов

Давайте заглянем внутрь!

In [ ]:
# ============================================================
# ШАГ 11: Извлекаем и визуализируем attention weights
# ============================================================

# Функция для получения attention weights
def get_attention_weights(text, model, tokenizer, layer=-1):  # функция получения внимания
    # Получаем attention weights из модели
    inputs = tokenizer(text, return_tensors='pt').to(device)  # токенизируем
    with torch.no_grad():                            # без градиентов
        outputs = model(**inputs, output_attentions=True)  # запрашиваем attention

    # outputs.attentions - кортеж тензоров (по одному на слой)
    # Каждый тензор: [batch, num_heads, seq_len, seq_len]
    attentions = outputs.attentions                  # все внимания

    if layer == -1:                                  # если последний слой
        layer_attn = attentions[-1]                  # берём последний слой
    else:                                            # иначе
        layer_attn = attentions[layer]               # берём указанный слой

    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])  # токены

    return layer_attn[0].cpu().numpy(), tokens       # возвращаем внимания и токены

# Пример: смотрим на attention для предложения
text = "Кот сидит на диване и спал спокойно"         # пример текста
attn_weights, tokens = get_attention_weights(text, model, tokenizer)  # получаем внимания

print(f"Текст: {text}")
print(f"Токены: {tokens}")
print(f"Форма attention: {attn_weights.shape}")      # [12 голов, токены, токены]

# Визуализируем все 12 голов внимания
fig, axes = plt.subplots(3, 4, figsize=(20, 14))    # сетка 3x4
fig.suptitle(f'Attention Heads (последний слой)\nТекст: "{text}"', fontsize=16, y=1.02)  # заголовок

for head_idx in range(12):                           # перебираем 12 голов
    row = head_idx // 4                              # строка
    col = head_idx % 4                               # столбец
    ax = axes[row, col]                              # текущий подграфик

    # Рисуем тепловую карту
    im = ax.imshow(attn_weights[head_idx], cmap='YlOrRd', aspect='auto')  # тепловая карта
    ax.set_xticks(range(len(tokens)))                # метки X
    ax.set_yticks(range(len(tokens)))                # метки Y
    ax.set_xticklabels(tokens, rotation=90, fontsize=7)  # подписи X
    ax.set_yticklabels(tokens, fontsize=7)           # подписи Y
    ax.set_title(f'Голова {head_idx+1}', fontsize=11)  # заголовок

plt.tight_layout()                                   # компоновка
plt.show()                                           # показываем

In [ ]:
# ============================================================
# ШАГ 11 (продолжение): Визуализация конкретного attention head
# ============================================================

# Увеличенная визуализация одной головы внимания
# Выберем голову, которая лучше всего показывает синтаксические связи

text_probe = "Мальчик который играл во дворе пришёл домой"  # предложение с подчинительной связью
attn_probe, tokens_probe = get_attention_weights(text_probe, model, tokenizer)  # внимания

# Ищем голову с максимальным вниманием к синтаксическим связям
# (простая эвристика: наибольшая концентрация внимания)
best_head = 0                                        # лучшая голова
max_concentration = 0                                # максимальная концентрация

for h in range(12):                                  # перебираем головы
    attn = attn_probe[h]                             # внимание головы
    # Считаем "концентрацию": насколько внимание сфокусировано
    max_attn = attn.max(axis=1).mean()               # средний макс внимание
    if max_attn > max_concentration:                 # если больше
        max_concentration = max_attn                 # обновляем
        best_head = h                                # запоминаем

print(f"Лучшая голова для синтаксических связей: {best_head + 1}")

# Увеличенная визуализация
fig, ax = plt.subplots(figsize=(12, 10))             # создаём фигуру

# Рисуем тепловую карту для лучшей головы
sns.heatmap(attn_probe[best_head],                   # данные
            xticklabels=tokens_probe,                # метки X
            yticklabels=tokens_probe,                # метки Y
            cmap='YlOrRd',                           # цветовая карта
            annot=True,                              # показываем числа
            fmt='.2f',                               # формат чисел
            annot_kws={'size': 9},                   # размер шрифта
            ax=ax,                                   # оси
            linewidths=0.5)                          # линии между ячейками

ax.set_title(f'Attention Head {best_head + 1}: синтаксические связи\n'
             f'Текст: "{text_probe}"', fontsize=14)  # заголовок
ax.set_xlabel('Ключ (Key)', fontsize=12)             # ось X
ax.set_ylabel('Запрос (Query)', fontsize=12)         # ось Y

plt.tight_layout()                                   # компоновка
plt.show()                                           # показываем

print("\nЯркие ячейки показывают, на какие токены модель обращает больше внимания.")
print("Обычно модель связывает: прилагательные с существительными,")
print("подлежащие со сказуемыми, местоимения с их антецедентами.")

In [ ]:
# ============================================================
# ШАГ 11 (продолжение): Визуализация по слоям - что каждый слой изучает
# ============================================================

# Визуализируем одну голову на разных слоях
text_layers = "Старый кот спокойно спал на мягком диване"  # пример текста
inputs_l = tokenizer(text_layers, return_tensors='pt').to(device)  # токенизируем

with torch.no_grad():                                # без градиентов
    outputs_l = model(**inputs_l, output_attentions=True)  # все внимания

tokens_l = tokenizer.convert_ids_to_tokens(inputs_l['input_ids'][0])  # токены

# Выбираем голову 0 и визуализируем на слоях 0, 3, 6, 11
selected_layers = [0, 3, 6, 11]                      # выбранные слои
head_to_show = 0                                     # голова для показа

fig, axes = plt.subplots(2, 2, figsize=(16, 14))    # сетка 2x2
fig.suptitle(f'Attention Head 1 на разных слоях\nТекст: "{text_layers}"',
             fontsize=16, y=1.02)                    # заголовок

for idx, layer in enumerate(selected_layers):        # перебираем слои
    row = idx // 2                                   # строка
    col = idx % 2                                    # столбец
    ax = axes[row, col]                              # текущий подграфик

    attn = outputs_l.attentions[layer][0, head_to_show].cpu().numpy()  # внимание

    sns.heatmap(attn,                                # данные
                xticklabels=tokens_l,                # метки X
                yticklabels=tokens_l,                # метки Y
                cmap='YlOrRd',                       # цвета
                ax=ax,                               # оси
                linewidths=0.5,                      # линии
                annot=False)                         # без чисел

    layer_desc = {                                   # описание слоёв
        0: "Нижний слой: локальные связи (соседние слова)",
        3: "Средний слой: синтаксические связи",
        6: "Верхний слой: семантические связи",
        11: "Последний слой: глубокие семантические связи",
    }
    ax.set_title(f'Слой {layer+1}: {layer_desc.get(layer, "")}', fontsize=11)  # заголовок

plt.tight_layout()                                   # компоновка
plt.show()                                           # показываем

print("Наблюдение: нижние слои фокусируются на соседних словах,")
print("а верхние - на смысловых связях между далёкими словами.")

---
## Шаг 12. Практические советы для BERT

### 1. Learning Rate

| LR | Когда использовать |
|----|-------------------|
| **2e-5** | Стандартный выбор для fine-tuning |
| **3e-5** | Для маленьких датасетов (< 10K) |
| **5e-5** | Для очень маленьких датасетов (< 1K) |
| **1e-5** | Для больших датасетов, осторожная настройка |

### 2. Batch Size

| BS | Плюсы | Минусы |
|----|-------|--------|
| **32** | Стабильные градиенты | Много памяти GPU |
| **16** | Баланс | Стандартный выбор |
| **8** | Мало памяти | Шумные градиенты |
| **4** | Минимум памяти | Очень шумно |

### 3. Заморозка слоёв

Можно заморозить часть слоёв BERT (не обновлять их веса):

```
Замороженные слои (не обучаются)   Обучаемые слои
+---+                              +---+
| 1 |  <- заморожен                | 9 |
| 2 |  <- заморожен                |10 |
| 3 |  <- заморожен                |11 |
| 4 |  <- заморожен                |12 |
|...|                              |Cla| <- классификатор
+---+                              +---+
```

> Заморозка ускоряет обучение и предотвращает переобучение!

In [ ]:
# ============================================================
# ШАГ 12: Демонстрация заморозки слоёв BERT
# ============================================================

# Создаём новую модель для эксперимента
freeze_model = BertForSequenceClassification.from_pretrained(  # модель
    model_name,                                      # название
    num_labels=2                                     # 2 класса
)
freeze_model.to(device)                              # на устройство

# Вариант 1: Замораживаем все слои BERT (обучаем только классификатор)
print("Вариант 1: Все слои BERT заморожены (обучаем только классификатор)")
for param in freeze_model.bert.parameters():        # перебираем параметры BERT
    param.requires_grad = False                      # замораживаем

trainable_1 = sum(p.numel() for p in freeze_model.parameters() if p.requires_grad)  # обучаемые
total_1 = sum(p.numel() for p in freeze_model.parameters())  # всего
print(f"  Обучаемые параметры: {trainable_1:,} ({100*trainable_1/total_1:.2f}%)")

# Вариант 2: Замораживаем нижние 6 слоёв
freeze_model_2 = BertForSequenceClassification.from_pretrained(  # модель
    model_name, num_labels=2
)
freeze_model_2.to(device)                            # на устройство

print("\nВариант 2: Нижние 6 слоёв заморожены")
for i, layer in enumerate(freeze_model_2.bert.encoder.layer):  # перебираем слои
    if i < 6:                                       # первые 6 слоёв
        for param in layer.parameters():            # перебираем параметры
            param.requires_grad = False              # замораживаем

trainable_2 = sum(p.numel() for p in freeze_model_2.parameters() if p.requires_grad)  # обучаемые
print(f"  Обучаемые параметры: {trainable_2:,} ({100*trainable_2/total_1:.2f}%)")

# Вариант 3: Все слои обучаются (стандартный fine-tuning)
print("\nВариант 3: Все слои обучаются (стандартный fine-tuning)")
trainable_3 = total_1                                # все параметры обучаемы
print(f"  Обучаемые параметры: {trainable_3:,} (100.00%)")

# Визуализация сравнения
fig, ax = plt.subplots(figsize=(10, 6))              # создаём фигуру

variants = ['Заморожен весь BERT\n(только классификатор)',
            'Заморожено 6/12 слоёв',
            'Все слои обучаются\n(стандартный fine-tuning)']  # варианты
trainable_pcts = [100*trainable_1/total_1, 100*trainable_2/total_1, 100.0]  # проценты
bar_colors = ['#3498db', '#f39c12', '#e74c3c']       # цвета

bars = ax.bar(variants, trainable_pcts, color=bar_colors, edgecolor='black', linewidth=1.5)  # столбцы

for bar, pct in zip(bars, trainable_pcts):           # подписываем
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,  # позиция
            f'{pct:.2f}%', ha='center', fontsize=12, fontweight='bold')  # текст

ax.set_ylabel('Обучаемые параметры (%)', fontsize=14)  # ось Y
ax.set_title('Стратегии заморозки слоёв BERT', fontsize=16)  # заголовок
ax.set_ylim(0, 110)                                 # пределы по Y

plt.tight_layout()                                   # компоновка
plt.show()                                           # показываем

print("\nРекомендация: начните с заморозки нижних слоёв,")
print("затем постепенно размораживайте, если точности не хватает.")

In [ ]:
# ============================================================
# ШАГ 12 (продолжение): Визуализация влияния learning rate
# ============================================================

# Демонстрация: как learning rate влияет на обучение
# Симулируем разные learning rates

lrs = [1e-6, 2e-5, 5e-5, 1e-4, 1e-3]               # разные learning rates
lr_labels = ['1e-6 (очень мало)', '2e-5 (стандарт)', '5e-5 (агрессивно)',
             '1e-4 (очень агрессивно)', '1e-3 (слишком много)']  # метки

# Симулируем loss curves (типичное поведение)
np.random.seed(42)                                   # фиксируем seed
steps = 100                                          # количество шагов

fig, ax = plt.subplots(figsize=(12, 6))              # создаём фигуру

for lr, label in zip(lrs, lr_labels):                # перебираем LR
    # Симулируем: маленький LR -> медленно, оптимальный -> быстро, большой -> нестабильно
    if lr < 1e-5:                                   # слишком маленький
        loss = 1.5 * np.exp(-0.01 * np.arange(steps)) + 0.5 + np.random.randn(steps) * 0.02  # медленно
    elif lr < 3e-5:                                  # оптимальный
        loss = 1.5 * np.exp(-0.05 * np.arange(steps)) + 0.2 + np.random.randn(steps) * 0.02  # хорошо
    elif lr < 1e-4:                                  # агрессивный
        loss = 1.5 * np.exp(-0.08 * np.arange(steps)) + 0.15 + np.random.randn(steps) * 0.05  # быстрее, но шумно
    elif lr < 5e-4:                                  # очень агрессивный
        loss = 1.5 * np.exp(-0.1 * np.arange(steps)) + 0.3 + np.random.randn(steps) * 0.1  # нестабильно
    else:                                            # слишком большой
        loss = np.concatenate([                       # расходится
            1.5 * np.exp(-0.15 * np.arange(30)) + 0.5,
            0.5 + np.cumsum(np.random.randn(70) * 0.1)  # расходится
        ])

    ax.plot(range(steps), loss, label=label, linewidth=2)  # рисуем

ax.set_xlabel('Шаг обучения', fontsize=14)           # ось X
ax.set_ylabel('Loss', fontsize=14)                   # ось Y
ax.set_title('Влияние Learning Rate на обучение BERT', fontsize=16)  # заголовок
ax.legend(fontsize=11)                               # легенда
ax.set_ylim(0, 2.5)                                 # пределы по Y

plt.tight_layout()                                   # компоновка
plt.show()                                           # показываем

print("Оптимальный LR для BERT: 2e-5 ... 5e-5")
print("Слишком маленький -> обучение не сходится")
print("Слишком большой -> обучение расходится")

---
## Шаг 13. Практические задания

Теперь ваша очередь! Вот 5 заданий для закрепления материала.

> Каждое задание проверяет понимание разных аспектов BERT.

---

### Задание 1: Исследование MLM на русском языке

Создайте 10 предложений с `[MASK]` на разные темы:
- Наука, спорт, музыка, природа, технологии
- Для каждого предложения получите топ-5 предсказаний
- Определите: какие предсказания правильные? Какие неожиданные?
- Выведите среднюю вероятность правильного предсказания

```python
# Ваш код здесь
```

---

### Задание 2: Сравнение контекстных embeddings

Выберите 3 слова с несколькими значениями (как "банк"):
- Для каждого слова создайте по 4 предложения с разными контекстами
- Получите embeddings токена из BERT
- Визуализируйте через t-SNE
- Проанализируйте: группируются ли одинаковые значения?

```python
# Ваш код здесь
```

---

### Задание 3: Fine-tuning с заморозкой слоёв

Дообучите BERT для классификации с разным количеством замороженных слоёв:
- Вариант A: заморожено 0 слоёв (все обучаются)
- Вариант B: заморожено 6 слоёв
- Вариант C: заморожено 9 слоёв
- Сравните: точность, время обучения, использование памяти
- Постройте графики сравнения

```python
# Ваш код здесь
```

---

### Задание 4: Пробинг attention heads

Проведите собственный анализ attention heads:
- Возьмите 5 предложений разной структуры
- Для каждого предложения визуализируйте все 12 голов последнего слоя
- Определите, какие головы отвечают за:
  - Связь прилагательное + существительное
  - Связь подлежащее + сказуемое
  - Связь местоимение + антецедент
- Запишите свои наблюдения

```python
# Ваш код здесь
```

---

### Задание 5: BERT для новой задачи

Выберите одну из задач и реализуйте fine-tuning:
- **Вариант A**: Классификация на 3+ классов (например: позитивный/нейтральный/негативный)
- **Вариант B**: Определение тематики текста (спорт/политика/технологии/культура)
- **Вариант C**: Определение языка текста (русский/английский/французский)

Шаги:
1. Создайте датасет (минимум 50 примеров на класс)
2. Разбейте на train/val/test
3. Дообучите BERT
4. Оцените качество (accuracy, F1, confusion matrix)
5. Проанализируйте ошибки модели

```python
# Ваш код здесь
```

---

### Критерии оценки

| Задание | Баллы | Что проверяется |
|---------|-------|-----------------|
| 1 | 15 | Умение работать с MLM и анализировать предсказания |
| 2 | 20 | Понимание контекстных embeddings |
| 3 | 25 | Навык fine-tuning с оптимизацией |
| 4 | 20 | Умение интерпретировать attention |
| 5 | 20 | Способность применить BERT к новой задаче |

**Итого: 100 баллов**

---
## Итоги

### Что мы изучили

| Тема | Ключевой вывод |
|------|---------------|
| **BERT** | Двунаправленный энкодер, предобученный на MLM + NSP |
| **WordPiece** | Умная токенизация: частые слова целиком, редкие - по кускам |
| **Архитектура** | 12 слоёв, 768 размерность, 12 голов внимания, 110M параметров |
| **Embeddings** | Контекстные: одно слово -> разные векторы в разных контекстах |
| **MLM** | Интерактивное предсказание замаскированных слов |
| **Fine-tuning** | Быстрое дообучение под конкретную задачу |
| **Attention** | Разные головы специализируются на разных типах связей |
| **Практика** | LR=2e-5, batch=16, заморозка слоёв для оптимизации |

### Следующие шаги

1. Попробуйте **RuBERT** (модель специально для русского языка)
2. Исследуйте **DistilBERT** (быстрее, легче, почти так же хорошо)
3. Изучите **GPT** - генеративную модель на Transformer
4. Попробуйте **T5** - модель для text-to-text задач

> BERT - это фундамент современного NLP. Освоив его, вы готовы к любым задачам обработки текста!

## FastAPI-сервер для BERT

Эндпоинты:

- `POST /classify` - классификация текста
- `POST /fill_mask` - заполнение [MASK]
- `POST /embed` - получить эмбеддинг текста

Доступ через ngrok.

In [ ]:
# Установка зависимостей
!pip install fastapi uvicorn pyngrok transformers torch -q

In [ ]:
# Импорты для BERT FastAPI
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List, Optional
import uvicorn
import threading
import torch
from transformers import pipeline, AutoTokenizer, AutoModel

# Загружаем модели (берём небольшие для скорости)
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME)

# Pipeline для mask filling
fill_mask = pipeline('fill-mask', model=MODEL_NAME)
print(f'Модель {MODEL_NAME} загружена.')

In [ ]:
# FastAPI приложение для BERT
app = FastAPI(title='BERT API', version='1.0')

class ClassifyRequest(BaseModel):
    text: str
    labels: List[str] = ['positive', 'negative', 'neutral']

class FillMaskRequest(BaseModel):
    text: str
    top_k: int = 5

class EmbedRequest(BaseModel):
    text: str

@app.post('/embed')
def get_embed(req: EmbedRequest):
    # Получить эмбеддинг через BERT (mean pooling)
    inputs = tokenizer(req.text, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    last_hidden = outputs.last_hidden_state[0]
    embedding = last_hidden.mean(dim=0).numpy()
    return {'text': req.text, 'dim': int(embedding.shape[0]), 'embedding': embedding.tolist()}

@app.post('/fill_mask')
def fill_mask_endpoint(req: FillMaskRequest):
    # Заполнить [MASK] в тексте
    text = req.text
    if '[MASK]' not in text:
        text = text + ' [MASK].'
    results = fill_mask(text, top_k=req.top_k)
    return {
        'input': text,
        'predictions': [{'token': r['token_str'], 'score': float(r['score']), 'sequence': r['sequence']} for r in results]
    }

@app.post('/classify')
def classify_text(req: ClassifyRequest):
    # Эвристическая классификация на основе эмбеддинга (демо)
    # В реальном приложении используйте fine-tuned BERTClassifier
    positive_words = ['good', 'great', 'excellent', 'love', 'happy', 'best']
    negative_words = ['bad', 'terrible', 'hate', 'sad', 'worst', 'awful']
    text_lower = req.text.lower()
    pos_score = sum(1 for w in positive_words if w in text_lower)
    neg_score = sum(1 for w in negative_words if w in text_lower)
    if pos_score > neg_score:
        label = 'positive'
    elif neg_score > pos_score:
        label = 'negative'
    else:
        label = 'neutral'
    confidence = abs(pos_score - neg_score) / max(1, pos_score + neg_score + 1)
    return {
        'text': req.text,
        'label': label,
        'confidence': round(confidence, 4),
        'scores': {'positive': pos_score, 'negative': neg_score}
    }

@app.get('/health')
def health():
    return {'status': 'ok', 'model': MODEL_NAME}

print('FastAPI приложение BERT готово.')

In [ ]:
# Запуск через ngrok
from pyngrok import ngrok
public_url = ngrok.connect(8000)
print(f'API: {public_url}')
print(f'Docs: {public_url}/docs')

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='error')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
import time; time.sleep(3)
print('Сервер запущен.')

In [ ]:
# Тестирование BERT API
import requests
base = str(public_url)

# 1. Fill mask
r = requests.post(f'{base}/fill_mask', json={'text': 'The capital of France is [MASK].', 'top_k': 3})
print('Fill mask results:')
for p in r.json()['predictions']:
    print(f'  [{p["score"]:.4f}] {p["sequence"]}')

# 2. Classify
r = requests.post(f'{base}/classify', json={'text': 'I love this amazing product, it is the best!'})
print('\nClassification:', r.json())

# 3. Embed
r = requests.post(f'{base}/embed', json={'text': 'Hello world'})
print(f'\nEmbedding dim: {r.json()["dim"]}')

## Интерактивные виджеты для BERT

Настройте параметры через dropdown и слайдеры.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

model_dd = widgets.Dropdown(
    options=['distilbert-base-uncased', 'bert-base-uncased', 'bert-base-multilingual-cased'],
    value='distilbert-base-uncased',
    description='Модель BERT:'
)
text_input = widgets.Text(value='The weather today is [MASK].', description='Текст:')
top_k_slider = widgets.IntSlider(value=5, min=1, max=10, step=1, description='top_k:')
fill_btn = widgets.Button(description='Fill [MASK]', button_style='primary', icon='magic')
classify_btn = widgets.Button(description='Классифицировать', button_style='success', icon='tag')
out = widgets.Output()

def on_fill(b):
    with out:
        clear_output()
        print(f'Модель: {model_dd.value}')
        try:
            r = requests.post(f'{str(public_url)}/fill_mask', json={'text': text_input.value, 'top_k': top_k_slider.value})
            print(f'\nПредсказания:')
            for p in r.json()['predictions']:
                print(f'  [{p["score"]:.4f}] {p["sequence"]}')
        except Exception as e:
            print(f'Ошибка: {e}. Запустите сервер выше.')

def on_classify(b):
    with out:
        clear_output()
        try:
            r = requests.post(f'{str(public_url)}/classify', json={'text': text_input.value})
            data = r.json()
            print(f'Label: {data["label"]} (confidence: {data["confidence"]})')
            print(f'Scores: {data["scores"]}')
        except Exception as e:
            print(f'Ошибка: {e}. Запустите сервер выше.')

fill_btn.on_click(on_fill)
classify_btn.on_click(on_classify)
display(model_dd, text_input, top_k_slider, fill_btn, classify_btn, out)